# SegFormer flood susceptibility — Colab GPU training

**Before running:** upload these to a folder called `swat_flood` in your Google Drive:
- `code/segformer_flood.py`
- `data/processed/feature_cube.npy`  (2.7 GB)
- `data/processed/valid_mask.npy`
- `data/labels/flood_labels.tif`

Then: **Runtime → Change runtime type → GPU (T4)** and run all cells.
Training takes roughly 1–3 h on a T4; results are copied back to Drive at the end.

In [ ]:
# 1. Mount Google Drive and verify the uploads are there
from google.colab import drive
drive.mount('/content/drive')
import os
SRC = '/content/drive/MyDrive/swat_flood'
for f in ['segformer_flood.py', 'feature_cube.npy', 'valid_mask.npy', 'flood_labels.tif']:
    p = os.path.join(SRC, f)
    print(('OK  ' if os.path.exists(p) else 'MISSING  ') + p)

In [ ]:
# 2. Install the one missing dependency and check the GPU
!pip -q install rasterio
import torch
print('CUDA available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only!')

In [ ]:
# 3. Copy data to local disk (much faster than training off Drive)
!mkdir -p /content/work/data/processed /content/work/data/labels /content/work/runs
!cp "{SRC}/segformer_flood.py" /content/work/
!cp "{SRC}/feature_cube.npy" "{SRC}/valid_mask.npy" /content/work/data/processed/
!cp "{SRC}/flood_labels.tif" /content/work/data/labels/
%cd /content/work
!ls -lh data/processed data/labels

In [ ]:
# 4. Import the training module and point its config at the local copies
import sys; sys.path.insert(0, '/content/work')
import segformer_flood as sf
sf.CFG.update({
    'cube_path':  'data/processed/feature_cube.npy',
    'valid_path': 'data/processed/valid_mask.npy',
    'label_path': 'data/labels/flood_labels.tif',
    'out_dir':    'runs/segformer',
    'num_workers': 2,
})
print(sf.DEVICE)  # should say cuda

In [ ]:
# 5. Train (early stopping will end it well before 100 epochs)
model, cube, valid = sf.train(sf.CFG)

In [ ]:
# 6. Full-basin susceptibility map from the best checkpoint
prob = sf.predict_map(model, cube, valid, sf.CFG)

In [ ]:
# 7. Copy everything back to Drive (checkpoint, history, susceptibility map)
!mkdir -p "{SRC}/results"
!cp runs/segformer/best_model.pt runs/segformer/history.json runs/segformer/susceptibility.npy "{SRC}/results/"
print('Done — download the results folder from Drive into paper/runs/segformer/ on your PC.')